# 第 03 期：主成分分析与多源环境变量降维

本 notebook 使用教学构造数据演示 PCA 的基本流程：构造相关环境变量、标准化、拟合 PCA、解释方差、载荷分析和低维可视化。示例数据不代表真实区域观测结果。


## 1. 导入库并设置路径


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

EPISODE_DIR = Path.cwd()
if not (EPISODE_DIR / "figures").exists() and Path("episodes/03-pca/figures").exists():
    EPISODE_DIR = Path("episodes/03-pca")

DATA_DIR = EPISODE_DIR / "data"
FIGURE_DIR = EPISODE_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## 2. 构造多源环境变量教学数据

这里用三个潜在梯度构造变量：水分梯度、地形梯度和温度梯度。这样可以得到“相关但不完全重复”的多源环境变量结构。


In [ ]:
n_samples = 180

moisture_gradient = rng.normal(0, 1, n_samples)
terrain_gradient = rng.normal(0, 1, n_samples)
temperature_gradient = rng.normal(0, 1, n_samples)
noise = lambda scale: rng.normal(0, scale, n_samples)

precip = 760 + 145 * moisture_gradient + 25 * terrain_gradient + noise(38)
temp = 14.5 + 2.2 * temperature_gradient - 1.4 * terrain_gradient - 0.002 * precip + noise(0.45)
elevation = 1450 + 430 * terrain_gradient - 80 * temperature_gradient + noise(90)
slope = 14 + 5.5 * terrain_gradient + noise(2.2)
soil_moisture = 0.42 + 0.105 * moisture_gradient - 0.018 * terrain_gradient + noise(0.035)
ndvi = 0.50 + 0.085 * moisture_gradient + 0.035 * temperature_gradient - 0.020 * terrain_gradient + noise(0.035)
aridity = 1.15 - 0.18 * moisture_gradient + 0.10 * temperature_gradient + noise(0.055)

soil_moisture = np.clip(soil_moisture, 0.05, 0.85)
ndvi = np.clip(ndvi, 0.05, 0.90)
aridity = np.clip(aridity, 0.35, 1.85)
slope = np.clip(slope, 1, 36)

data = pd.DataFrame({
    "sample_id": np.arange(1, n_samples + 1),
    "precip": precip,
    "temp": temp,
    "elevation": elevation,
    "slope": slope,
    "ndvi": ndvi,
    "soil_moisture": soil_moisture,
    "aridity": aridity,
})

conditions = [
    data["elevation"] < data["elevation"].quantile(0.33),
    data["elevation"] < data["elevation"].quantile(0.66),
]
choices = ["lowland", "hillslope"]
data["terrain_group"] = np.select(conditions, choices, default="upland")

data = data[[
    "sample_id", "terrain_group", "precip", "temp", "elevation", "slope",
    "ndvi", "soil_moisture", "aridity"
]]

data.to_csv(DATA_DIR / "environment_pca_demo.csv", index=False)
data.head()


## 3. 原始变量相关性

先看变量之间是否存在相关结构。PCA 适合用于概括相关变量的共同变化方向，但相关矩阵本身不能证明地学机制。


In [ ]:
features = [
    "precip", "temp", "elevation", "slope", "ndvi", "soil_moisture", "aridity"
]

corr = data[features].corr()

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha="right")
ax.set_yticklabels(features)

for i in range(len(features)):
    for j in range(len(features)):
        color = "white" if abs(corr.iloc[i, j]) > 0.55 else "black"
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color=color, fontsize=9)

cbar = fig.colorbar(im, ax=ax, shrink=0.82)
cbar.set_label("Correlation")
ax.set_title("Correlation Matrix of Environmental Variables")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "environment_correlation_heatmap.png", dpi=180, bbox_inches="tight")
plt.close(fig)

corr.round(2)


## 4. 标准化并拟合 PCA

多源环境变量单位不同，常见做法是先标准化，再进行 PCA。是否标准化取决于研究目的；本例采用标准化以避免量纲主导结果。


In [ ]:
X = data[features]

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

pca = PCA()
scores = pca.fit_transform(X_std)

score_columns = [f"PC{i+1}" for i in range(len(features))]
scores_df = pd.DataFrame(scores, columns=score_columns)
scores_df["sample_id"] = data["sample_id"]
scores_df["terrain_group"] = data["terrain_group"]

loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=score_columns,
)

explained_df = pd.DataFrame({
    "component": score_columns,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_),
})

explained_df.round(3)


## 5. 解释方差图

解释方差比例说明每个主成分保留了输入变量中多少方差信息。不要把解释方差直接理解为预测能力或机制强度。


In [ ]:
x = np.arange(1, len(features) + 1)

fig, ax1 = plt.subplots(figsize=(8, 5.4))
ax1.bar(x, pca.explained_variance_ratio_, color="#4C78A8", alpha=0.85, label="Individual")
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance Ratio")
ax1.set_xticks(x)
ax1.set_xticklabels(score_columns)
ax1.set_ylim(0, max(pca.explained_variance_ratio_) * 1.25)

ax2 = ax1.twinx()
ax2.plot(x, np.cumsum(pca.explained_variance_ratio_), color="#F58518", marker="o", label="Cumulative")
ax2.set_ylabel("Cumulative Explained Variance")
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
ax1.set_title("PCA Explained Variance")
ax1.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "explained_variance.png", dpi=180, bbox_inches="tight")
plt.close(fig)


## 6. PC1-PC2 样本散点图

PCA 是无监督方法，不使用 `terrain_group` 标签。这里的颜色只是为了辅助观察样本在低维空间中的分布。


In [ ]:
group_colors = {
    "lowland": "#54A24B",
    "hillslope": "#E45756",
    "upland": "#4C78A8",
}

fig, ax = plt.subplots(figsize=(7.2, 6))
for group, group_df in scores_df.groupby("terrain_group"):
    ax.scatter(
        group_df["PC1"], group_df["PC2"],
        s=42, alpha=0.78, label=group, color=group_colors[group], edgecolor="white", linewidth=0.5
    )

ax.axhline(0, color="gray", linewidth=0.8, alpha=0.5)
ax.axvline(0, color="gray", linewidth=0.8, alpha=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Samples Projected onto PC1 and PC2")
ax.legend(title="Terrain group", frameon=True)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pc_scores_scatter.png", dpi=180, bbox_inches="tight")
plt.close(fig)


## 7. 主成分载荷图

载荷帮助理解主成分由哪些原始变量组合而成。主成分整体符号可以翻转，因此解释时应关注变量之间的相对方向和组合。


In [ ]:
pc12 = loadings[["PC1", "PC2"]]

fig, ax = plt.subplots(figsize=(8.5, 5.4))
bar_width = 0.38
positions = np.arange(len(features))
ax.bar(positions - bar_width / 2, pc12["PC1"], width=bar_width, label="PC1", color="#4C78A8")
ax.bar(positions + bar_width / 2, pc12["PC2"], width=bar_width, label="PC2", color="#F58518")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(features, rotation=35, ha="right")
ax.set_ylabel("Loading")
ax.set_title("Variable Loadings on PC1 and PC2")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_loadings.png", dpi=180, bbox_inches="tight")
plt.close(fig)

pc12.round(3)


## 8. PCA 双标图

双标图把样本得分和变量载荷方向放在同一张图中。它适合探索结构，但只展示 PC1 和 PC2 所包含的信息。


In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 6.4))
for group, group_df in scores_df.groupby("terrain_group"):
    ax.scatter(
        group_df["PC1"], group_df["PC2"],
        s=35, alpha=0.55, label=group, color=group_colors[group], edgecolor="white", linewidth=0.4
    )

arrow_scale = 3.2
for variable in features:
    x_loading = loadings.loc[variable, "PC1"] * arrow_scale
    y_loading = loadings.loc[variable, "PC2"] * arrow_scale
    ax.arrow(0, 0, x_loading, y_loading, color="#333333", alpha=0.85,
             head_width=0.08, head_length=0.10, length_includes_head=True)
    ax.text(x_loading * 1.08, y_loading * 1.08, variable, fontsize=9, ha="center", va="center")

ax.axhline(0, color="gray", linewidth=0.8, alpha=0.5)
ax.axvline(0, color="gray", linewidth=0.8, alpha=0.5)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA Biplot: Scores and Loadings")
ax.legend(title="Terrain group", loc="upper right")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_biplot.png", dpi=180, bbox_inches="tight")
plt.close(fig)


## 9. 输出结果表

下面输出解释方差和前两个主成分载荷，供文章或报告引用。


In [ ]:
print("Explained variance ratio:")
print(explained_df.round(3).to_string(index=False))

print("\nLoadings for PC1 and PC2:")
print(loadings[["PC1", "PC2"]].round(3).to_string())


## 10. 方法边界

本示例只演示 PCA 的基本流程。真实研究中，需要先处理空间投影、分辨率、时间窗口、掩膜、缺失值和异常值。若 PCA 被用于预测流程，应只在训练集上拟合标准化器和 PCA，再应用到验证集或测试集，避免数据泄漏。
